In [ ]:
# running this on kaggle

!sudo apt update && sudo apt install -y pciutils lshw
!sudo apt update && sudo apt install zstd -y
!curl -fsSL https://ollama.com/install.sh | sh

!curl -sSL https://ngrok-agent.s3.amazonaws.com/ngrok.asc \
  | sudo tee /etc/apt/trusted.gpg.d/ngrok.asc >/dev/null \
  && echo "deb https://ngrok-agent.s3.amazonaws.com bookworm main" \
  | sudo tee /etc/apt/sources.list.d/ngrok.list \
  && sudo apt update \
  && sudo apt install ngrok
!pip install pyngrok

In [ ]:
# --- MINISTRAL SETUP ---
import os
import subprocess
import time
from pyngrok import ngrok
from kaggle_secrets import UserSecretsClient

# 0. HARD CLEANUP: Stop Qwen/old processes immediately
print("Stopping previous model and clearing VRAM...")
!pkill -9 ollama || true
!pkill -9 ngrok || true
!sync; echo 3 > /proc/sys/vm/drop_caches
time.sleep(3)

# 1. Start Temp Server for Build
temp_env = os.environ.copy()
temp_env["OLLAMA_HOST"] = "127.0.0.1:11435"
proc = subprocess.Popen(["ollama", "serve"], env=temp_env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(7)

# 2. Create Ministral Modelfile
with open("Modelfile_Min", "w") as f:
    f.write("""FROM ministral-3:14b
PARAMETER num_gpu 999
PARAMETER num_ctx 16384
PARAMETER use_mmap false""")

print("Building Ministral (ministral-custom)...")
subprocess.run(["ollama", "create", "ministral-custom", "-f", "Modelfile_Min"], env=temp_env)

# 3. Shutdown Temp & Launch Final
proc.terminate()
proc.wait()

user_secrets = UserSecretsClient()
ngrok.set_auth_token(user_secrets.get_secret("ngrok-token"))
public_url = ngrok.connect(11434)
print(f"\n🚀 MINISTRAL READY\nURL: {public_url.public_url}")

!OLLAMA_NUM_CTX=16384 OLLAMA_HOST=0.0.0.0:11434 OLLAMA_ORIGINS="*" OLLAMA_SCHED_SPREAD=1 OLLAMA_KEEP_ALIVE=-1 ollama serve
